# Day 18: Building a Career Conversation Agent

In this notebook, we'll build a career chatbot that:
- Uses resources (LinkedIn profile, personal summary) to answer questions
- Uses tools to record user interest and unknown questions
- Can be deployed to your website

This is a practical implementation of AI agents using direct API calls (no frameworks).


## Step 1: Import Required Libraries

We'll need several libraries for this project:
- `openai`: To interact with OpenAI's GPT models
- `python-dotenv`: To load environment variables from .env file
- `PyPDF2`: To read PDF files (LinkedIn profile)
- `gradio`: To create a simple web interface
- `requests`: To send push notifications


In [3]:
# Import necessary libraries
from dotenv import load_dotenv
import os
from openai import OpenAI
from PyPDF2 import PdfReader
import gradio as gr
import requests
import json


## Step 2: Load Environment Variables

We'll load our API keys from the .env file.
The .env file should contain:
- OPENAI_API_KEY: Your OpenAI API key
- PUSHOVER_USER: Your Pushover user key (optional)
- PUSHOVER_TOKEN: Your Pushover API token (optional)


In [4]:
# Load environment variables from .env file
# override=True ensures .env values take priority
load_dotenv(override=True)

# Check if OpenAI API key is loaded
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key:
    print(f"✓ OpenAI API key loaded (starts with: {openai_key[:10]}...)")
else:
    print("✗ OpenAI API key not found. Please check your .env file.")


✓ OpenAI API key loaded (starts with: sk-proj-Tk...)


## Step 3: Initialize OpenAI Client

Create an instance of the OpenAI client that we'll use to make API calls.


In [5]:
# Initialize OpenAI client
# This is a lightweight library for connecting to OpenAI's API
openai = OpenAI()


## Step 4: Set Up Pushover for Notifications

Pushover allows us to send push notifications to our phone.
This is useful for knowing when users want to connect or ask questions we can't answer.

To set up Pushover:
1. Go to https://pushover.net and create an account
2. Install the Pushover app on your phone
3. Get your User Key and create an API Token
4. Add them to your .env file


In [6]:
# Pushover configuration
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"
PUSHOVER_USER = os.getenv("PUSHOVER_USER", "")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN", "")

def push(message):
    """
    Send a push notification to your phone via Pushover.
    
    Args:
        message (str): The message to send
    """
    if not PUSHOVER_USER or not PUSHOVER_TOKEN:
        print(f"[Pushover not configured] Would send: {message}")
        return
    
    try:
        response = requests.post(PUSHOVER_URL, data={
            "user": PUSHOVER_USER,
            "token": PUSHOVER_TOKEN,
            "message": message
        })
        if response.status_code == 200:
            print(f"✓ Notification sent: {message}")
        else:
            print(f"✗ Failed to send notification: {response.status_code}")
    except Exception as e:
        print(f"✗ Error sending notification: {e}")

# Test the push function (optional)
# Uncomment the line below to test
# push("Test notification from Career Agent!")

print("✓ Pushover configured")


✓ Pushover configured


## Step 5: Define Tool Functions

These are the actual Python functions that our agent will be able to call.
We have two tools:
1. `record_user_details`: Records when someone wants to get in touch
2. `record_unknown_question`: Records questions the agent couldn't answer


In [7]:
def record_user_details(email, name="", notes=""):
    """
    Record when a user wants to get in touch.
    Sends a push notification with the user's details.
    
    Args:
        email (str): User's email address (required)
        name (str): User's name (optional)
        notes (str): Additional notes (optional)
    
    Returns:
        str: Confirmation message
    """
    message = f"📧 Recording interest from {name} ({email}): {notes}"
    push(message)
    return "Recorded successfully. Thank you for your interest!"

def record_unknown_question(question):
    """
    Record a question the agent couldn't answer.
    Sends a push notification so we can improve the agent.
    
    Args:
        question (str): The question that couldn't be answered
    
    Returns:
        str: Confirmation message
    """
    message = f"❓ Recording question I couldn't answer: {question}"
    push(message)
    return "Question recorded for future improvement."

print("✓ Tool functions defined")


✓ Tool functions defined


## Step 6: Define Tools in JSON Format

We need to describe our tools in JSON format so the LLM understands:
- What each tool does
- When to use it
- What parameters it needs

This JSON is what gets sent to OpenAI's API.


In [8]:
# Tool definition for recording user details
record_user_details_json = {
    "type": "function",
    "function": {
        "name": "record_user_details",
        "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {
                    "type": "string",
                    "description": "The user's email address"
                },
                "name": {
                    "type": "string",
                    "description": "The user's name if provided"
                },
                "notes": {
                    "type": "string",
                    "description": "Any additional notes or context about the user's interest"
                }
            },
            "required": ["email"],
            "additionalProperties": False
        }
    }
}

# Tool definition for recording unknown questions
record_unknown_question_json = {
    "type": "function",
    "function": {
        "name": "record_unknown_question",
        "description": "Use this tool when you don't know the answer to a question. Record the question so it can be answered later.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The question that couldn't be answered"
                }
            },
            "required": ["question"],
            "additionalProperties": False
        }
    }
}

# Combine tools into a list
tools = [
    record_user_details_json,
    record_unknown_question_json
]

print("✓ Tools defined successfully")
print(f"  - {tools[0]['function']['name']}")
print(f"  - {tools[1]['function']['name']}")

# This completes Step 6


✓ Tools defined successfully
  - record_user_details
  - record_unknown_question


## Step 7: Implement Tool Call Handler

This function executes the tools that the LLM wants to use.
It's the "glorified if statement" that makes tool calling work!


In [9]:
def handle_tool_calls(tool_calls):
    """
    Execute the tools the LLM wants to use.
    
    Args:
        tool_calls: List of tool calls from the LLM response
    
    Returns:
        List of messages with tool results
    """
    messages = []
    
    for tool_call in tool_calls:
        # Extract tool information
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"🔧 Calling tool: {tool_name}")
        
        # Use Python's globals() to dynamically call the function
        # This is more elegant than a series of if statements
        tool_function = globals()[tool_name]
        result = tool_function(**arguments)
        
        # Add result to messages in the format OpenAI expects
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
    
    return messages

print("✓ Tool handler defined")


✓ Tool handler defined


## Step 8: Load Career Resources

Now we'll load information about your career:
- LinkedIn profile (PDF)
- Personal summary (text file)

**Important:** Replace these files with your own!
- Create a folder called `me/`
- Add your `linkedin.pdf` (download from LinkedIn)
- Add your `summary.txt` (write a brief summary about yourself)


In [12]:
# Read LinkedIn PDF
# Note: You should replace this with your own LinkedIn PDF!
try:
    reader = PdfReader("me/linkedin.pdf")
    linkedin_text = ""
    for page in reader.pages:
        linkedin_text += page.extract_text()
    print(f"✓ LinkedIn profile loaded ({len(linkedin_text)} characters)")
except FileNotFoundError:
    print("✗ LinkedIn PDF not found. Using placeholder text.")
    linkedin_text = """[Your LinkedIn profile would go here. 
    Download your profile as PDF from LinkedIn and place it in me/linkedin.pdf]"""

# Read personal summary
# Note: You should replace this with your own summary!
try:
    with open("me/summary.txt", "r", encoding="utf-8") as f:
        summary = f.read()
    print(f"✓ Summary loaded ({len(summary)} characters)")
except FileNotFoundError:
    print("✗ Summary file not found. Using placeholder text.")
    summary = """[Your personal summary would go here. 
    Create a file me/summary.txt with a brief summary about yourself, 
    including fun facts and personal touches.]"""

# Set your name
# IMPORTANT: Change this to your actual name!
name = "Qasim Dawood"

print(f"\n✓ Resources loaded for {name}")


✓ LinkedIn profile loaded (11332 characters)
✓ Summary loaded (813 characters)

✓ Resources loaded for Qasim Dawood


## Step 9: Create System Prompt

The system prompt sets the context for the agent:
- Who it's representing (you!)
- What information it has (resources)
- How it should behave
- When to use tools


In [13]:
# Create the system prompt with resources
system_prompt = f"""
You are acting as {name}.

You are answering questions on that person's website, particularly questions 
related to their career, background, skills, and experience.

Your responsibility is to represent {name} for interactions on the website 
as faithfully as possible.

You are given a summary of their background and their LinkedIn profile.
Use this information to answer questions accurately and professionally.

Be professional and engaging. If you don't know the answer to a question, 
say so clearly and use your tool to record the unknown question.

If the user is engaging in discussion and seems interested, try to steer 
them towards getting in touch via email. When they provide their email, 
record it using your tool.

## Summary
{summary}

## LinkedIn Profile
{linkedin_text}

With this context, please chat with the user, always staying in character as {name}.
"""

print("✓ System prompt created")
print(f"  Length: {len(system_prompt)} characters")


✓ System prompt created
  Length: 13016 characters


## Step 10: Create the Chat Function with Agent Loop

This is the heart of our agent!
It implements the agent loop:
1. Call LLM with message and available tools
2. If LLM wants to use a tool, execute it
3. Call LLM again with tool results
4. Repeat until LLM is done
5. Return final response


In [19]:
def chat(message, history):
    """
    Main chat function with tool calling support.
    
    Args:
        message (str): Current user message
        history (list): Previous conversation as list of tuples [(user, bot), ...]
    
    Returns:
        str: Agent's response
    """
    # Build messages list
    # Start with system prompt
    messages = [{"role": "system", "content": system_prompt}]
    
    # Add history - Gradio provides it as list of tuples [(user_msg, bot_msg), ...]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    
    # Add current message
    messages.append({"role": "user", "content": message})
    
    # Agent loop - keep going until done
    done = False
    while not done:
        # Call OpenAI with tools
        response = openai.chat.completions.create(
            model="gpt-4o-mini",  # You can change this to any model
            messages=messages,
            tools=tools  # Our tool descriptions
        )
        
        # Check what the LLM wants to do
        finish_reason = response.choices[0].finish_reason
        
        if finish_reason == "tool_calls":
            # LLM wants to use tools
            print("\n🤖 LLM wants to use tools...")
            
            # Get the tool calls
            tool_calls = response.choices[0].message.tool_calls
            
            # Add LLM's message (including tool requests) to messages
            messages.append(response.choices[0].message)
            
            # Execute the tools
            tool_results = handle_tool_calls(tool_calls)
            
            # Add tool results to messages
            messages.extend(tool_results)
            
            # Loop back to call LLM again with results
        else:
            # LLM is done, return the response
            done = True
            return response.choices[0].message.content

print("✓ Chat function defined")


✓ Chat function defined


## Step 11: Create Gradio Interface

Gradio makes it incredibly easy to create a chat interface.
Just pass our chat function and launch!


In [20]:
# Create Gradio chat interface
interface = gr.ChatInterface(
    fn=chat,
    title=f"Career Conversation with {name}",
    description="Ask me about my career, experience, and background!",
    examples=[
        "What's your current role?",
        "What's your greatest achievement?",
        "What technologies do you work with?",
        "Tell me about a challenge you overcame",
        "I'd like to get in touch"
    ]
)

# Launch the interface
# This will open a web browser with your chat interface
print("\n🚀 Launching Gradio interface...")
print("   The interface will open in your browser.")
print("   Press Ctrl+C to stop.\n")

interface.launch()


c:\Users\HP\.conda\envs\llm-env\Lib\site-packages\gradio\chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(



🚀 Launching Gradio interface...
   The interface will open in your browser.
   Press Ctrl+C to stop.

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Testing the Agent

Try asking these questions to test different features:

**Testing Resources:**
- "What's your current job?"
- "What's your educational background?"
- "Tell me about your experience"

**Testing Unknown Question Tool:**
- "What's your favorite color?"
- "Who's your favorite musician?"
- Any question not in your resources

**Testing User Details Tool:**
- "I'd like to get in touch"
- Then provide your email when asked

Watch the console output to see when tools are being called!


## Next Steps

**To Deploy This Agent:**
1. Save this code as `app.py` (convert from notebook)
2. Run `gradio deploy` in your terminal
3. Follow the prompts to deploy to Hugging Face Spaces
4. Embed in your website!

**To Improve This Agent:**
1. Add more resources (projects, achievements, fun facts)
2. Implement RAG for better context retrieval
3. Add more tools (database queries, calendar integration)
4. Add an evaluator to check response quality
5. Improve the UI with custom Gradio themes
6. Add streaming responses for better UX

**To Learn More:**
- Experiment with different models
- Try different prompt engineering techniques
- Add your own custom tools
- Share your agent and get feedback!


## Conclusion

Congratulations! You've built a real AI agent from scratch using:
- **Resources**: Your LinkedIn profile and summary
- **Tools**: Functions the LLM can call
- **Agent Loop**: Iterative execution until goal achieved

This is the foundation for all AI agents. Next week, we'll see how frameworks
make this easier, but you now understand what's happening under the hood!

**Key Takeaways:**
1. Tool calling is "just" JSON and if statements
2. Resources turn generic LLMs into domain experts
3. The agent loop (LLM + tools + loop) is remarkably effective
4. You can build production-ready agents with direct API calls

Keep building and experimenting! 🚀
